# PRICING DINÁMICO DE HABITACIONES DE HOTEL

*Cristian Rubio Barato*

*Francisco Martínez Esteso*

*José Vicente García López*

*Víctor Ortega Gómez*


## ÍNDICE

1. [PREPROCESAMIENTO](#1-preprocesamiento)  

2. [ENTRENAMIENTO](#2-entrenamiento)  

3. [MONITORIZACION](#3-monitorizacion)  

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, r2_score, mean_absolute_error, mean_squared_error, max_error
from sklearn.model_selection import train_test_split
import numpy as np
from datetime import datetime
import joblib
from prometheus_client import start_http_server, Gauge
import psutil
import threading
import time


In [ ]:
random_state = 42
np.random.seed(42)

## 1. PREPROCESAMIENTO

In [ ]:
df = pd.read_csv('hotel_booking.csv')

categorical_vars = df.select_dtypes(include=['object','category']).columns.tolist()
continuous_vars  = df.select_dtypes(include=['int64','float64']).columns.tolist()

df = df.dropna(subset=['adr'])
df.fillna(0, inplace=True)
df['children'] = df['children'].astype(int)

df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'])
df['reservation_status_year'] = df['reservation_status_date'].dt.year
df['arrival_date_month'] = pd.to_datetime(df['arrival_date_month'], format='%B').dt.month
df['reservation_status_date'] = df['reservation_status_date'].dt.date

start_date = datetime.strptime('2015-07','%Y-%m').date()
df = df[df['reservation_status_date'] >= start_date]

cols_to_drop = [
    "days_in_waiting_list","name","email","phone-number","credit_card",
    "agent","company","booking_changes","arrival_date_week_number"
]
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

df = df[~((df['adults']==0)&(df['children']==0)&(df['babies']==0))]

Q1, Q3 = np.percentile(df['adr'],[25,75])
IQR = Q3 - Q1
df = df[(df['adr']>=Q1-1.5*IQR)&(df['adr']<=Q3+1.5*IQR)]

categorical_vars = [c for c in categorical_vars if c not in cols_to_drop]
df = pd.get_dummies(df, columns=categorical_vars, drop_first=True)

numerical_cols = df.select_dtypes(include=['int64','float64']).columns
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])


In [9]:
y = df["adr"]
X = df.drop(columns=["adr"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

## 2. ENTRENAMIENTO

In [ ]:
metrics = {
    "R2": "r2",
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "Max Error": "max_error"
}

model = RandomForestRegressor(random_state=42)
param_grid = {
    "n_estimators": [100],
    "max_depth": [None],
    "min_samples_split": [5]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=metrics,
    refit="R2",
    cv=5,
    return_train_score=True,
    verbose=1
)

grid_search.fit(X_train, y_train)

y_pred = grid_search.predict(X_test)

print("Métricas en test:")
print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("Max Error:", max_error(y_test, y_pred))

joblib.dump(grid_search.best_estimator_, "best_rf_model.pkl")
np.save("y_test.npy", y_test)
np.save("y_pred.npy", y_pred)


Fitting 5 folds for each of 1 candidates, totalling 5 fits
Métricas en test:
R2: 0.8577353899701413
MAE: 0.19435527926290613
MSE: 0.14187309000519974
RMSE: 0.37666044390830283
Max Error: 4.5699378348444935


## 3. MONITORIZACION

In [ ]:
# Definir métricas Prometheus del sistema
cpu_usage = Gauge('system_cpu_usage_percent', 'CPU usage percent')
ram_usage = Gauge('system_ram_usage_percent', 'RAM usage percent')

# Definir métricas Prometheus del modelo
model_r2_score = Gauge('model_r2_score', 'R2 score of trained model')
model_mae = Gauge('model_mae', 'Mean Absolute Error of trained model')
model_mse = Gauge('model_mse', 'Mean Squared Error of trained model')
model_rmse = Gauge('model_rmse', 'Root Mean Squared Error of trained model')
model_max_error = Gauge('model_max_error', 'Max Error of trained model')

# Función para actualizar métricas
def monitor_metrics():
    while True:
        # Métricas de sistema
        cpu_usage.set(psutil.cpu_percent())
        ram_usage.set(psutil.virtual_memory().percent)

        y_true = np.load("y_test.npy")
        y_pred = np.load("y_pred.npy")

        # Métricas de modelo
        model_r2_score.set(r2_score(y_true, y_pred))
        model_mae.set(mean_absolute_error(y_true, y_pred))
        model_mse.set(mean_squared_error(y_true, y_pred))
        model_rmse.set(np.sqrt(mean_squared_error(y_true, y_pred)))
        model_max_error.set(max_error(y_true, y_pred))

        time.sleep(5)

# Iniciar servidor Prometheus
start_http_server(8000)
threading.Thread(target=monitor_metrics, daemon=True).start()

print("Servidor: http://localhost:8000/metrics")
    

Servidor: http://localhost:8000/metrics
